In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
jax.numpy.array(1, dtype=int)

import mmml
import ase
import numpy as np

from mmml.interfaces.pycharmmInterface.mm_system_energy import (
    CharmmNbondSettings,
    mm_system_energy_and_forces,
)

from pathlib import Path
from mmml.interfaces.pycharmmInterface.import_pycharmm import ensure_pycharmm_loaded
ensure_pycharmm_loaded()
from mmml.interfaces.pycharmmInterface.trialanine_water_box import build_trialanine_water_box_in_charmm
box = build_trialanine_water_box_in_charmm(n_waters=200, box_side_A=28.0, seed=11, workdir=Path('/tmp/tria_box'))
print(len(box.positions), box.psf_path)# ase.Atoms()
positions = box.positions
from mmml.interfaces.pycharmmInterface.import_pycharmm import CGENFF_PRM
prm = CGENFF_PRM
from mmml.interfaces.pycharmmInterface.import_pycharmm import pycharmm_loud
import pycharmm.lingo as lingo

pycharmm_loud()

from mmml.interfaces.pycharmmInterface.cgenff_bonded_reference import (
    charmm_bonded_forces_kcalmol_A,
    charmm_nonbonded_energy_components_kcalmol,
    run_charmm_nonbonded_ener_force,
    set_charmm_positions,
    setup_nonbonded_only_charmm,
)
from mmml.interfaces.pycharmmInterface.mm_system_energy import (
    load_nonbonded_system_from_charmm,
    nonbonded_energy_and_forces,
)


pos = np.asarray(positions, dtype=np.float64)
pos = np.random.uniform(-4.1, 4.1, pos.shape) + pos
set_charmm_positions(pos)

print("pos", pos)

setup_nonbonded_only_charmm()
print("setup_nonbonded_only_charmm")
# perform charmm minimization
lingo.charmm_script("""CONStraint DROPlet FORC 0.01 EXPO 4 ! [FORCe real] [EXPOnent integer] [NOMAss]""")
from mmml.interfaces.pycharmmInterface.charmm_levels import run_charmm_script_loud
run_charmm_script_loud("""
    MINI SD 10000
""")
# print("run_charmm_script_loud")
lingo.charmm_script("""CONStraint DROPlet !""")
from mmml.interfaces.pycharmmInterface.charmm_levels import run_charmm_script_loud
run_charmm_script_loud("""
    MINI SD 10000
""")
print("run_charmm_script_loud")
# lingo.charmm_script("""CONStraint DROPlet FORC 0.01 EXPO 1 ! [FORCe real] [EXPOnent integer] [NOMAss]""")
from mmml.interfaces.pycharmmInterface.import_pycharmm import coor
pos = coor.get_positions()[["x", "y", "z"]].to_numpy(dtype=float)
print("pos", pos)
from mmml.interfaces.pycharmmInterface.utils import get_Z_from_psf
z = get_Z_from_psf()
print("z", z)
atoms = ase.Atoms(z, pos)
print("atoms", atoms)
# save atoms to pdb file
atoms.write("atoms.pdb")

run_charmm_nonbonded_ener_force(silent=False)
charmm_terms = charmm_nonbonded_energy_components_kcalmol()
charmm_forces = charmm_bonded_forces_kcalmol_A()

# print(charmm_terms)
psf_path = box.psf_path
prm_path = CGENFF_PRM
cell = box.cell
from mmml.interfaces.pycharmmInterface.charmm_jax_energy_benchmark import _nbond_settings_from_cutoffs
nb_settings = _nbond_settings_from_cutoffs(box.nbond_cutoffs)
nbond_data = load_nonbonded_system_from_charmm(psf_path, prm_path)
jax_terms_raw, jax_forces = nonbonded_energy_and_forces(
    pos,
    nbond_data,
    cell,
    nb_settings,
)
jax_terms = {k: float(v) for k, v in jax_terms_raw.items()}
jax_forces_np = np.asarray(jax_forces, dtype=np.float64)

print(jax_terms)
# print(jax_forces_np)
print(charmm_terms)
# print(charmm_forces)

import matplotlib.pyplot as plt
plt.matshow(charmm_forces[:42])
plt.matshow(charmm_forces[42:])
plt.matshow(jax_forces[:42])
plt.matshow(jax_forces[42:])
plt.matshow(charmm_forces[:42] - jax_forces[:42])
plt.matshow(charmm_forces[42:] - jax_forces[42:])

# 0.1


In [ ]:

lingo.charmm_script(
    """
DEFINE PHSSOLU SELE RESNAME TRIA .AND. .NOT. HYDROGEN END
DEFINE PHSWAT  SELE RESNAME TIP3 .AND. TYPE OH2 END

SHOW NSEL SELE PHSSOLU END
SHOW NSEL SELE PHSWAT END
"""
)

In [ ]:
import pycharmm.psf as psf

print(set(psf.get_res()))
print(set(psf.get_atype()))

In [ ]:
jax.devices()

In [ ]:
from mmml.interfaces.pycharmmInterface import import_pycharmm
import_pycharmm.view_pycharmm_state()

In [ ]:
lingo.charmm_script("""ENERGY
""")

In [ ]:
lingo.charmm_script("""CONStraint DROPlet !FORC 0.001 ! [FORCe real] [EXPOnent integer] [NOMAss]""")

In [ ]:
# print("run_charmm_script_loud")
# lingo.charmm_script("""CONStraint DROPlet !""")
from mmml.interfaces.pycharmmInterface.charmm_levels import run_charmm_script_loud
run_charmm_script_loud("""
    MINI SD 1000
""")
lingo.charmm_script("MINI SD 1000")
print("run_charmm_script_loud")
# lingo.charmm_script("""CONStraint DROPlet FORC 0.01 EXPO 1 ! [FORCe real] [EXPOnent integer] [NOMAss]""")
from mmml.interfaces.pycharmmInterface.import_pycharmm import coor
pos = coor.get_positions()[["x", "y", "z"]].to_numpy(dtype=float)
print("pos", pos)
from mmml.interfaces.pycharmmInterface.utils import get_Z_from_psf
z = get_Z_from_psf()
print("z", z)
atoms = ase.Atoms(z, pos)
print("atoms", atoms)
# save atoms to pdb file
atoms.write("atoms.pdb")

In [ ]:
molecule_id = np.empty(len(atoms), dtype=np.int32)

# Trialanine
molecule_id[:42] = 0

# 50 water molecules
for mol_id, start in enumerate(
    range(42, len(atoms), 3),
    start=1,
):
    molecule_id[start:start + 3] = mol_id

assert np.all(molecule_id[:42] == 0)

for start in range(42, len(atoms), 3):
    assert len(set(molecule_id[start:start + 3])) == 1

# assert molecule_id.max() == 50

In [ ]:
jax_terms_raw, jax_forces = nonbonded_energy_and_forces(
    pos,
    nbond_data,
    cell,
    nb_settings,
    molecule_id=molecule_id,
)

In [ ]:
full_terms, _ = nonbonded_energy_and_forces(
    pos,
    nbond_data,
    cell,
    nb_settings,
)

inter_terms, _ = nonbonded_energy_and_forces(
    pos,
    nbond_data,
    cell,
    nb_settings,
    molecule_id=molecule_id,
)

In [ ]:
print("full:", {k: float(v) for k, v in full_terms.items()})
print("inter:", {k: float(v) for k, v in inter_terms.items()})

In [ ]:
charmm_terms

In [ ]:
jax_terms_raw

In [ ]:
nb_settings

# 0.2


In [ ]:
from mmml.interfaces.pycharmmInterface import import_pycharmm
import_pycharmm.view_pycharmm_state()

In [ ]:
CKPT_PATH = "params_aaa_long_2026-07-04_22-30-27.json"

In [ ]:
from mmml.interfaces.calculators.simple_inference import create_calculator_from_checkpoint

In [ ]:
calc = create_calculator_from_checkpoint(CKPT_PATH)

In [ ]:
calc

In [ ]:
prot = atoms[:42]
prot.get_atomic_numbers()
_pos = prot.get_positions()
prot.set_positions(_pos - _pos.T.mean(axis=-1))

In [ ]:
water = atoms[42:]
water.get_atomic_numbers()
_pos = water.get_positions()
# water.set_positions(_pos - _pos.T.mean(axis=-1))

In [ ]:
ase.visualize.view(prot, viewer="x3d")

In [ ]:
ase.visualize.view(water, viewer="x3d")

In [ ]:
ase.visualize.view(water + prot, viewer="x3d")

In [ ]:
prot_water = prot + water

In [ ]:
prot.calc = calc
water.calc = calc
prot_water.calc = calc

In [ ]:
prot.get_potential_energy(), water.get_potential_energy(), prot_water.get_potential_energy()

In [ ]:
prot.get_forces()

In [ ]:
from ase.optimize import QuasiNewton

# 3. Attach to optimizer (QuasiNewton is an alias for BFGSLineSearch)
dyn = QuasiNewton(prot, trajectory='prot_min.traj')
# 4. Run minimization until max force on any atom < 0.05 eV/Å
dyn.run(fmax=0.3)


In [ ]:
ase.visualize.view(prot, viewer="x3d")

In [ ]:
# 3. Attach to optimizer (QuasiNewton is an alias for BFGSLineSearch)
dyn = QuasiNewton(water, trajectory='water_min.traj')
# 4. Run minimization until max force on any atom < 0.05 eV/Å
dyn.run(fmax=5)
ase.visualize.view(water, viewer="x3d")

In [ ]:
prot_water = prot + water 
prot_water.calc = calc

In [ ]:
# 3. Attach to optimizer (QuasiNewton is an alias for BFGSLineSearch)
dyn = QuasiNewton(prot_water, trajectory='prot_water_min.traj')
# 4. Run minimization until max force on any atom < 0.05 eV/Å
dyn.run(fmax=3)


In [ ]:
ase.visualize.view(prot_water, viewer="x3d")

In [ ]:
z = get_Z_from_psf()


In [ ]:
# pos = prot_water.get_positions()

# set_charmm_positions(pos)
pos = prot_water.get_positions()
print("pos", pos)

setup_nonbonded_only_charmm()
print("setup_nonbonded_only_charmm")
# perform charmm minimization
from mmml.interfaces.pycharmmInterface.charmm_levels import run_charmm_script_loud
run_charmm_script_loud("""
    MINI SD 10000
    MINI ABNR 10000
""")
print("run_charmm_script_loud")
from mmml.interfaces.pycharmmInterface.import_pycharmm import coor
pos = coor.get_positions()[["x", "y", "z"]].to_numpy(dtype=float)
print("pos", pos)
from mmml.interfaces.pycharmmInterface.utils import get_Z_from_psf
z = get_Z_from_psf()
print("z", z)
atoms = ase.Atoms(z, pos)
print("atoms", atoms)

# save atoms to pdb file
atoms.write("atoms.pdb")

In [ ]:
from mmml.interfaces.pycharmmInterface import import_pycharmm
import_pycharmm.view_pycharmm_state()

In [ ]:
box.nbond_cutoffs

In [ ]:
from mmml.interfaces.pycharmmInterface.import_pycharmm import coor
pos = coor.get_positions()[["x", "y", "z"]].to_numpy(dtype=float)
print("pos", pos)
from mmml.interfaces.pycharmmInterface.utils import get_Z_from_psf
z = get_Z_from_psf()
print("z", z)
atoms = ase.Atoms(z, pos)
print("atoms", atoms)

# save atoms to pdb file
atoms.write("atoms.pdb")

run_charmm_nonbonded_ener_force(silent=False)
charmm_terms = charmm_nonbonded_energy_components_kcalmol()
charmm_forces = charmm_bonded_forces_kcalmol_A()

# print(charmm_terms)
psf_path = box.psf_path
prm_path = CGENFF_PRM
cell = box.cell
from mmml.interfaces.pycharmmInterface.charmm_jax_energy_benchmark import _nbond_settings_from_cutoffs
nb_settings = _nbond_settings_from_cutoffs(box.nbond_cutoffs)
nbond_data = load_nonbonded_system_from_charmm(psf_path, prm_path)
jax_terms_raw, jax_forces = nonbonded_energy_and_forces(
    pos,
    nbond_data,
    cell,
    nb_settings,
        molecule_id=molecule_id,
)
jax_terms = {k: float(v) for k, v in jax_terms_raw.items()}
jax_forces_np = np.asarray(jax_forces, dtype=np.float64)

print(jax_terms)
# print(jax_forces_np)
print(charmm_terms)
# print(charmm_forces)

In [ ]:
jax_terms_raw, jax_forces = nonbonded_energy_and_forces(
    pos,
    nbond_data,
    cell,
    nb_settings,
    molecule_id=molecule_id,
)

In [ ]:
jax_terms_raw

In [ ]:
from ase.calculators.calculator import Calculator

In [ ]:
import copy
import numpy as np

from ase.calculators.calculator import Calculator, all_changes
from ase.calculators.mixing import SumCalculator
from ase.units import kcal, mol

class MolecularPhysNetCalculator(Calculator):
    implemented_properties = ["energy", "forces"]

    def __init__(
        self,
        peptide_calc,
        water_calc,
        peptide_indices,
        water_indices,
        **kwargs,
    ):
        super().__init__(**kwargs)

        self.peptide_calc = peptide_calc
        self.water_calc = water_calc

        self.peptide_indices = np.asarray(peptide_indices, dtype=int)
        self.water_indices = [
            np.asarray(idx, dtype=int)
            for idx in water_indices
        ]

    def calculate(
        self,
        atoms=None,
        properties=("energy", "forces"),
        system_changes=all_changes,
    ):
        super().calculate(atoms, properties, system_changes)

        energy = 0.0
        forces = np.zeros((len(atoms), 3), dtype=np.float64)

        # Trialanine
        peptide = atoms[self.peptide_indices].copy()
        peptide.pbc = False
        peptide.calc = self.peptide_calc

        energy += peptide.get_potential_energy()
        forces[self.peptide_indices] += peptide.get_forces()

        # Waters
        for idx in self.water_indices:
            water = atoms[idx].copy()
            water.pbc = False
            water.calc = self.water_calc

            energy += water.get_potential_energy()
            forces[idx] += water.get_forces()

        self.results = {
            "energy": float(energy),
            "forces": forces,
        }

In [ ]:
atomic_numbers = z
z = np.asarray(atoms.numbers)

n_trialanine = 42

assert (len(z) - n_trialanine) % 3 == 0
assert np.all(
    z[n_trialanine:].reshape(-1, 3)
    == np.array([8, 1, 1])
)

monomer_indices = [np.arange(n_trialanine)]

monomer_indices.extend(
    np.arange(i, i + 3)
    for i in range(n_trialanine, len(atoms), 3)
)

In [ ]:
calc = create_calculator_from_checkpoint(CKPT_PATH)

In [ ]:
class MonomerSumCalculator(Calculator):
    implemented_properties = ["energy", "forces"]

    def __init__(
        self,
        monomer_indices,
        calculators=None,
        calculator_factory=None,
        **kwargs,
    ):
        super().__init__(**kwargs)

        self.monomer_indices = [
            np.asarray(indices, dtype=int)
            for indices in monomer_indices
        ]

        if calculators is not None:
            if len(calculators) != len(self.monomer_indices):
                raise ValueError(
                    "Need exactly one calculator per monomer."
                )
            self.calculators = list(calculators)

        elif calculator_factory is not None:
            self.calculators = [
                calculator_factory()
                for _ in self.monomer_indices
            ]

        else:
            raise ValueError(
                "Provide calculators or calculator_factory."
            )

    def calculate(
        self,
        atoms=None,
        properties=("energy", "forces"),
        system_changes=all_changes,
    ):
        super().calculate(atoms, properties, system_changes)

        energy = 0.0
        forces = np.zeros((len(atoms), 3), dtype=np.float64)

        for indices, calc in zip(
            self.monomer_indices,
            self.calculators,
        ):
            monomer = atoms[indices].copy()

            # Usually desirable for a molecular ML potential:
            monomer.pbc = False

            monomer.calc = calc

            energy += monomer.get_potential_energy()
            forces[indices] += monomer.get_forces()

        self.results = {
            "energy": float(energy),
            "forces": forces,
        }

In [ ]:
physnet_monomers = MonomerSumCalculator(
    monomer_indices=monomer_indices,
    calculator_factory=lambda: create_calculator_from_checkpoint(CKPT_PATH),
)

In [ ]:
factor = 0.0433641153087705  # eV per kcal/mol

class JAXIntermolecularCalculator(Calculator):
    implemented_properties = ["energy", "forces"]

    def __init__(
        self,
        nbond_data,
        nb_settings,
        energy_factor=1.0 * factor,
        force_factor=1.0 * factor,
        **kwargs,
    ):
        super().__init__(**kwargs)

        self.nbond_data = nbond_data
        self.nb_settings = nb_settings
        self.energy_factor = float(energy_factor)
        self.force_factor = float(force_factor)

    def calculate(
        self,
        atoms=None,
        properties=("energy", "forces"),
        system_changes=all_changes,
    ):
        super().calculate(atoms, properties, system_changes)

        pos = np.asarray(
            atoms.get_positions(),
            dtype=np.float64,
        )
        cell = np.asarray(
            atoms.cell.array,
            dtype=np.float64,
        )

        terms_raw, forces = nonbonded_energy_and_forces(
            pos,
            self.nbond_data,
            cell,
            self.nb_settings,
        )

        terms = {
            key: float(value)
            for key, value in terms_raw.items()
        }

        # Replace this with the exact total key used by your function.
        if "total" in terms:
            energy = terms["total"]
        else:
            energy = sum(terms.values())

        self.results = {
            "energy": energy * self.energy_factor,
            "forces": (
                np.asarray(forces, dtype=np.float64)
                * self.force_factor
            ),
        }

        # Optional diagnostic information.
        self.results["jax_terms"] = terms

# 0.3

In [ ]:
atoms.set_cell(box.cell)
atoms.set_pbc(True)

jax_inter = JAXIntermolecularCalculator(
    nbond_data=nbond_data,
    nb_settings=nb_settings,
)

atoms.calc = SumCalculator([
    physnet_monomers,
    jax_inter,
])

In [ ]:
energy = atoms.get_potential_energy()
forces = atoms.get_forces()

In [ ]:
energy

In [ ]:
forces

In [ ]:
ase.visualize.view(atoms, viewer="x3d")

In [ ]:
# # 3. Attach to optimizer (QuasiNewton is an alias for BFGSLineSearch)
# dyn = QuasiNewton(atoms, trajectory='prot_sum_calc_water_min.traj')
# # 4. Run minimization until max force on any atom < 0.05 eV/Å
# dyn.run(fmax=3)
from ase.optimize import FIRE
# 3. Initialize the FIRE optimizer
opt = FIRE(atoms, 
           # logfile='fire_log.txt'
          )

# 4. Run the minimization
# fmax represents the maximum force allowed (in eV/Angstrom)
opt.run(fmax=.5)

In [ ]:
ase.visualize.view(atoms, viewer="x3d")

In [ ]:
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase import units
from ase.md.verlet import VelocityVerlet
from ase import Atoms

def printenergy(atoms: Atoms) -> None:
    """Function to print the potential, kinetic and total energy"""
    epot = atoms.get_potential_energy() / len(atoms)
    ekin = atoms.get_kinetic_energy() / len(atoms)
    temperature = ekin / (1.5 * units.kB)

    print(f'Energy per atom: Epot = {epot:.3f}eV  Ekin = {ekin:.3f}eV '
          f'(T={temperature:3.0f}K)  Etot = {epot+ekin:.3f}eV')

from ase import units
from ase.md.verlet import VelocityVerlet
# Set the momenta corresponding to T=300K
MaxwellBoltzmannDistribution(atoms, temperature_K=300)
dyn = VelocityVerlet(atoms, 0.5 * units.fs,     trajectory="md.traj")
# print starting energies
printenergy(atoms)
# print energies as system evolves
for i in range(10):
    dyn.run(1000)
    printenergy(atoms)
    MaxwellBoltzmannDistribution(atoms, temperature_K=300)
# dyn = VelocityVerlet(cu_cube, 5 * units.fs)

In [ ]:
ase.visualize.view(atoms, viewer="x3d")

In [ ]:
import pycharmm.cons_harm as cons_harm
# cons_harm?

In [ ]:
# command = """
# MMFP
# BHEL SELE SEGID TRIA .AND. .NOT. HYDROGEN END
# SHEL SELE RESNAME TIP3 .AND. TYPE OH2 END -
#      DRSH 8.0 RWEL 0.25 PFINAL 1.0 -
#      SCO 0 RELA 0.00001 UPDF 10 -
#      CHFR 1000 SPACE 1000000 CHCO 0.00001 -
#      FOCO1 3.0 FOCO2 3.0 CUT 2.0
# END
# """

In [ ]:
# run_charmm_script_loud(command)

In [ ]:
lingo.charmm_script("""CONStraint DROPlet FORC 0.01 EXPO 1 ! [FORCe real] [EXPOnent integer] [NOMAss]""")

In [ ]:
# pos = atoms.get_positions()

# set_charmm_positions(pos)
atoms_tracker = []
# print("pos", pos)
for i in range(100):
    # setup_nonbonded_only_charmm()
    print("setup_nonbonded_only_charmm")
    # perform charmm minimization
    from mmml.interfaces.pycharmmInterface.charmm_levels import run_charmm_script_loud
    run_charmm_script_loud("""
    MINI SD 10000
    MINI ABNR 10000
    """)
    
    # print("atoms", atoms)
    
    # save atoms to pdb file
    
    import sys
    import os
    
    os.environ['CHARMM_LIB_DIR'] = '/Users/ericboittier/mmml/setup/charmm'
    
    import pycharmm
    import pycharmm.dynamics as charm_dyn
    
    # kw = {"start": True, "restart": False, "iasvel": 1, "iunrea": 88}
    dyn = pycharmm.DynamicsScript(lang=True, restart=False, nstep=50000, timest=0.005,
                            firstt=298.0, finalt=298.0, tbath=298.0, tstruc=298.0,
                            teminc=0.0, twindh=0.0, twindl=0.0,
                            iasors=0, iasvel=1, ichecw=0, iscale=0, iscvel=0,
                            echeck=-1.0, nsavc=10, nsavv=0, ntrfrq=1000, isvfrq=1000,
                            iprfrq=50, nprint=10, ihtfrq=0, ieqfrq=1,
                            ilbfrq=0)
    dyn.run()
    
    # print("run_charmm_script_loud")
    from mmml.interfaces.pycharmmInterface.import_pycharmm import coor
    pos = coor.get_positions()[["x", "y", "z"]].to_numpy(dtype=float)
    # print("pos", pos)
    from mmml.interfaces.pycharmmInterface.utils import get_Z_from_psf
    z = get_Z_from_psf()
    # print("z", z)
    atoms_ = ase.Atoms(z, pos)
    atoms_.write(f"atoms_{i}.pdb")
    atoms_tracker.append(atoms_)

In [ ]:
from mmml.interfaces.pycharmmInterface import import_pycharmm
import_pycharmm.view_pycharmm_state()